In [0]:
# -----------------------------------------------------------------------------
# Notebook: 01_ingest_customers
#
# Purpose:
#     Ingest Customers from Azure SQL Database into the Bronze layer.
#
# Source:
#     dbo.Customers
#
# Target:
#     Bronze Delta
#
# Load Type:
#     Full Load (Commented)
#     Incremental load (Watermark)
# -----------------------------------------------------------------------------

In [0]:
%run ./../setup/01_sql_configuration

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
from pyspark.sql import Row

In [0]:
run_ts = datetime.now(timezone.utc)
pipeline_name = 'ingest_customers_bronze'

In [0]:
customers_bronze_path = f"{BRONZE_PATH}/customers"

In [0]:
watermark_record_df = spark.read \
    .format("delta") \
    .load(f"{METADATA_PATH}/pipeline_watermarking") \
    .filter(F.col("pipeline_name") == pipeline_name) \
    .first()

In [0]:
watermark = (datetime(2000, 1, 1)) if watermark_record_df is None else watermark_record_df["last_processed_timestamp"]

In [0]:
sql_query = f"""
SELECT *
FROM dbo.Customers C
WHERE
    C.updated_at >= '{watermark}'
"""

In [0]:
df = (
        spark.read
        .format("jdbc")
        .options(
            url = jdbc_url,
            # Full load
            # dbtable = "dbo.Customers",
            query = (sql_query),
            user = jdbc_user,
            password = jdbc_password
        )
        .load()
    )

In [0]:
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
print(df.count())

In [0]:
# Full Load
#df.write.format("delta").mode("overwrite").save(customers_bronze_path)

df_is_empty = df.isEmpty()

# Incremental Load
if not df_is_empty:
    df.write.format("delta").mode("append").save(customers_bronze_path)

In [0]:
if df_is_empty:
    print("No records to process")
    last_processed_timestamp = watermark
else:
    last_processed_timestamp = df \
        .agg(
            F.max("updated_at").alias("updated_at")
        ).first()["updated_at"]

In [0]:
watermark_update_df = spark.createDataFrame(
    [
        (
            pipeline_name,
            last_processed_timestamp,
            run_ts,
            "SUCCESS",
            run_ts,
            run_ts
        )
    ],
    [
        "pipeline_name",
        "last_processed_timestamp",
        "last_run_timestamp",
        "last_run_status",
        "created_at",
        "updated_at"
    ]
)

In [0]:
from delta.tables import DeltaTable

watermark_table = DeltaTable.forPath(
    spark,
    f"{METADATA_PATH}/pipeline_watermarking"
)

(
    watermark_table.alias("target")
    .merge(
        watermark_update_df.alias("source"),
        "target.pipeline_name = source.pipeline_name"
    )
    .whenMatchedUpdate(
        set={
            "last_processed_timestamp": "source.last_processed_timestamp",
            "last_run_timestamp": "source.last_run_timestamp",
            "last_run_status": "source.last_run_status",
            "updated_at": "source.updated_at"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
spark.read \
    .format("delta") \
    .load(f"{METADATA_PATH}/pipeline_watermarking") \
    .filter(F.col("pipeline_name") == pipeline_name) \
    .show(truncate=False)